In [ ]:
#import data
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [ ]:
data = yf.download(['GC=F','DX-Y.NYB'], start = '2020-01-01').dropna()
data['GC_return'] = data[('Close', 'GC=F')].pct_change() * 100
data['DXY_return'] = data[('Close', 'DX-Y.NYB')].pct_change() * 100

/tmp/ipykernel_5315/2244709436.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(['GC=F','DX-Y.NYB'], start = '2020-01-01').dropna()
[*********************100%***********************]  2 of 2 completed


In [ ]:
#เราจะสมมุติฐานตารางข้อมูล (สร้างข้อมูลจำลองเพื่อใช้เรียนรู้)
np.random.seed(42)
date_rang = pd.date_range(start='2023-01-01',periods=100,freq='D')
df = pd.DataFrame(index=date_rang)
df['DXY_return'] = np.random.normal(0, 0.01, 100)
df['GC_return'] = -0.8 *df['DXY_return'] + np.random.normal(0,0.005,100)

In [ ]:
#ทำการจัดเรียงข้อมูล Feture(x),Target(y)
x=df[['DXY_return']]
y=df['GC_return']

In [ ]:
#แบ่งข้อมูลเป็นตารางสอน(Train)   และตารางสอบ(Test)  *ห้ามสลับข้อมูลมั่ว*
train_size = int(len(df)*0.8)
x_train,x_test = x.iloc[:train_size],x.iloc[train_size:]
y_train,y_test = y.iloc[:train_size],y.iloc[train_size:]

In [ ]:
#เรียกใช้โมเดล ML และสั่งเรียนรู้ fit  จากข้อมูลในอดีต
model = LinearRegression()
model.fit(x_train,y_train)

LinearRegression()

In [ ]:
#สั่งให้ ai ทำนาย (predict) ข้อมูลในชุดสอบ (ข้อมูลที่มันไม่เคยเห็นมาก่อน)
df_test_result = pd.DataFrame(index=x_test.index)
df_test_result['Actual_Gold'] = y_test
df_test_result['Predict_Gold'] = model.predict(x_test)


In [ ]:
#แปลงผลทำนายเป็น สัญญาณซื้อขาย (trading signal) for backtest
#ถ้าราคาจริง ต่ำกว่าราคาทำนาย แสดงว่าให้ไป buy (1) ไม่งั้นถือเงินสด (0)
df_test_result['Signal'] = np.where(
    df_test_result['Actual_Gold'] < df_test_result['Predict_Gold'],
    1,
    0
)


In [ ]:
print(df_test_result.head(10))

            Actual_Gold  Predict_Gold  Signal
2023-03-22     0.004886      0.002218       0
2023-03-23    -0.007143     -0.002763       1
2023-03-24    -0.017178     -0.012443       1
2023-03-25     0.006559      0.004797       0
2023-03-26     0.005351      0.007304       1
2023-03-27     0.007584      0.004655       0
2023-03-28    -0.004957     -0.007585       0
2023-03-29    -0.002994     -0.002518       1
2023-03-30     0.000004      0.004896       1
2023-03-31    -0.011680     -0.004112       1


In [ ]:
# 1. สร้างคอลัมน์ผลตอบแทนของระบบเทรด (Strategy Return)
# คิดง่ายๆ: เอาสัญญาณวันนี้ (Signal) ไปคูณกับ ผลตอบแทนทองคำในวันพรุ่งนี้ (Actual_Gold)
# สาเหตุที่ต้องใช้ .shift(-1) เพราะเราได้สัญญาณวันนี้ จะไปได้กำไร/ขาดทุนจากราคาของวันพรุ่งนี้ครับ
df_test_result['Strategy_Return'] = df_test_result['Signal'] * df_test_result['Actual_Gold'].shift(-1)

# 2. คำนวณกำไรสะสม (Cumulative Return) แบบทบต้น ดูว่าเงินงอกไหม
df_test_result['Cumulative_Market'] = (1 + df_test_result['Actual_Gold']).cumprod() - 1
df_test_result['Cumulative_Strategy'] = (1 + df_test_result['Strategy_Return']).cumprod() - 1

# ดูผลลัพธ์สรุปกำไรสะสมตอนท้ายสุดของข้อมูล
print("กำไรสะสมของตลาดทองคำปกติ:", df_test_result['Cumulative_Market'].iloc[-2] * 100, "%")
print("กำไรสะสมของระบบ AI คุณ :", df_test_result['Cumulative_Strategy'].iloc[-2] * 100, "%")

กำไรสะสมของตลาดทองคำปกติ: -1.359303708363746 %
กำไรสะสมของระบบ AI คุณ : -2.23833808045798 %
